# Notebook 01 — PDF Extraction & Chunking

## Learning Goals
- See how PDF text is extracted
- Understand what chunks look like
- Compare fixed-size vs section-aware chunking
- This is the FOUNDATION for all 4 RAG types

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

from utils.pdf_extractor import (
    extract_text_by_page,
    extract_full_text,
    extract_sections,
    get_paper_metadata
)
from utils.chunker import (
    fixed_size_chunk,
    section_aware_chunk,
    display_chunks
)

print("✅ All imports ready!")

✅ All imports ready!


## Step 1: Paper Metadata
First let's see basic info about our paper.

In [2]:
# Paper metadata extraction
meta = get_paper_metadata("../data/paper.pdf")

print("=" * 60)
print("PAPER INFORMATION")
print("=" * 60)
print(f"Title:  {meta['title']}")
print(f"Pages:  {meta['num_pages']}")
print(f"File:   {meta['filename']}")
print()
print("First page preview:")
print("-" * 40)
print(meta['first_page_preview'])

✅ Paper: Iterative Multi-Granular RAG with Contextual Hierarchical Gr...
✅ Pages: 9
PAPER INFORMATION
Title:  Iterative Multi-Granular RAG with Contextual Hierarchical Graph
Pages:  9
File:   paper.pdf

First page preview:
----------------------------------------
Iterative Multi-Granular RAG with Contextual Hierarchical Graph
Yanli Hu1*, Teng Liu1*, Zhuangyi Zhou1, Weixin Zeng1†, Zhen Tan1, Xiang Zhao2
1National Key Laboratory of Information Systems Engineering, National University of Defense Technology, China
2National Key Laboratory of Big Data and Decision, National University of Defense Technology, China
{huyanli, liuteng20, zhouzhuangyi, zengweixin13, tanzhen08a, xiangzhao}@nudt.edu.cn
Abstract
Retrieval-Augmented Generation (RAG) enhances large lan


## Step 2: Extract Text Page by Page
See what raw text looks like from each page.

In [3]:
#page text extraction
pages = extract_text_by_page("../data/paper.pdf")

print(f"Total pages extracted: {len(pages)}")
print()

# Show first 3 pages preview
for page in pages[:3]:
    print(f"{'='*60}")
    print(f"PAGE {page['page']}")
    print(f"{'='*60}")
    print(page['text'][:400])
    print(f"... ({len(page['text'])} total chars)")
    print()

✅ Extracted 9 pages from paper.pdf
Total pages extracted: 9

PAGE 1
Iterative Multi-Granular RAG with Contextual Hierarchical Graph
Yanli Hu1*, Teng Liu1*, Zhuangyi Zhou1, Weixin Zeng1†, Zhen Tan1, Xiang Zhao2
1National Key Laboratory of Information Systems Engineering, National University of Defense Technology, China
2National Key Laboratory of Big Data and Decision, National University of Defense Technology, China
{huyanli, liuteng20, zhouzhuangyi, zengweixin13,
... (5033 total chars)

PAGE 2
2025), which significantly improve reasoning flexibility
through retrieve-generate-reflect cycles. However, these ap-
proaches remain fundamentally constrained by “local infor-
mation horizons”: their decision-making processes rely en-
tirely on isolated, incrementally acquired evidence, failing to
capture global associative structures within knowledge bases
and are prone to suboptimal retrieval p
... (6370 total chars)

PAGE 3
𝓌
Query
Context
+
(𝑒1
𝑡, 𝜑𝑒1
𝑡)
…
…
Fresh
Historical
(𝑒1, 𝜑𝑒1 )
Upda

## Step 3: Detect Sections
Academic papers have clear sections.
We detect them to enable Hierarchical RAG later.

In [4]:
sections = extract_sections("../data/paper.pdf")

print(f"Total sections found: {len(sections)}")
print()
print("=" * 60)

for section_name, content in sections.items():
    print(f"\n[{section_name}]")
    print(f"Length: {len(content)} chars")
    print(f"Preview: {content[:150]}...")
    print("-" * 40)

✅ Extracted 9 pages from paper.pdf
✅ Detected 11 sections:
   → Header
   → Abstract
   → Introduction
   → Related Work
   → Preliminaries
   → Methodology
   → Experimental Settings
   → Method
   → Ablation Study
   → Acknowledgments
   → References
Total sections found: 11


[Header]
Length: 435 chars
Preview: Iterative Multi-Granular RAG with Contextual Hierarchical Graph
Yanli Hu1*, Teng Liu1*, Zhuangyi Zhou1, Weixin Zeng1†, Zhen Tan1, Xiang Zhao2
1Nationa...
----------------------------------------

[Abstract]
Length: 1797 chars
Preview: Retrieval-Augmented Generation (RAG) enhances large lan-
guage models (LLMs) with external knowledge retrieval, im-
proving factual accuracy and knowl...
----------------------------------------

[Introduction]
Length: 6120 chars
Preview: Human intelligence excels through continuous learning and
dynamic knowledge integration, enabling effective naviga-
tion of complex and evolving envir...
----------------------------------------

[Related Work]

## Step 4: Fixed Size Chunking
Split the full text into equal-sized chunks.
This is what Traditional RAG uses.

⚠️ Problem: chunks have NO idea what section they're from!

In [6]:
#fix size chunking
full_text = extract_full_text("../data/paper.pdf")
fixed_chunks = fixed_size_chunk(full_text, chunk_size=500, chunk_overlap=50)

print(f"Total chunks: {len(fixed_chunks)}")
print()

# Show 3 sample chunks
for chunk in fixed_chunks[5:8]:
    print(f"{'='*60}")
    print(f"Chunk #{chunk['chunk_id']}")
    print(f"Length: {chunk['char_count']} chars")
    print(f"Section: {chunk.get('section', '❌ UNKNOWN - No section info!')}")
    print(f"Text:")
    print(chunk['text'])
    print()

✅ Extracted 9 pages from paper.pdf
✅ Total characters extracted: 44702
✅ Created 100 chunks
   Avg chunk size: 455 chars
   Min chunk size: 52 chars
   Max chunk size: 499 chars
Total chunks: 100

Chunk #5
Length: 495 chars
Section: ❌ UNKNOWN - No section info!
Text:
tion of complex and evolving environments. Developing AI
systems with analogous capabilities is critical in high-stakes
domains, where outcomes depend on factual accuracy, ver-
ifiable evidence chains, and rigorous reasoning. Large Lan-
*These authors contributed equally.
†Corresponding author
Copyright © 2026, Association for the Advancement of Artificial
Intelligence (www.aaai.org). All rights reserved.
Q: Who directed a 2006 film where Ron Perkins character plays the 
manager of a hotel?

Chunk #6
Length: 492 chars
Section: ❌ UNKNOWN - No section info!
Text:
manager of a hotel?
LLM Response:
I don’t know
Evidence
1
Sentence: Ron Perkins appeared in 
“The Prestige”…
Key Phrases: Ron Perkins; The 
Prestige; Hugh Jackma
Ev

## Step 5: Section-Aware Chunking
Now split by section — each chunk KNOWS where it's from.
This enables Hierarchical RAG!

✅ Each chunk has section label attached.

In [8]:
# aware chunking
section_chunks = section_aware_chunk(sections, chunk_size=500)

print(f"Total section-aware chunks: {len(section_chunks)}")
print()

# Show chunks from different sections
shown_sections = set()
for chunk in section_chunks:
    section = chunk.get('section', 'Unknown')
    if section not in shown_sections and len(shown_sections) < 4:
        print(f"{'='*60}")
        print(f"Chunk #{chunk['chunk_id']}")
        print(f"Section: ✅ {section}")
        print(f"Length: {chunk['char_count']} chars")
        print(f"Text preview:")
        print(chunk['text'][:200])
        print()
        shown_sections.add(section)

✅ Created 74 section-aware chunks
   Sections included:
   → Abstract: 4 chunks
   → Introduction: 13 chunks
   → Related Work: 12 chunks
   → Preliminaries: 8 chunks
   → Methodology: 17 chunks
   → Experimental Settings: 6 chunks
   → Method: 6 chunks
   → Ablation Study: 8 chunks
Total section-aware chunks: 74

Chunk #0
Section: ✅ Abstract
Length: 476 chars
Text preview:
Retrieval-Augmented Generation (RAG) enhances large lan-
guage models (LLMs) with external knowledge retrieval, im-
proving factual accuracy and knowledge coverage. However,
existing RAG approaches fa

Chunk #4
Section: ✅ Introduction
Length: 474 chars
Text preview:
Human intelligence excels through continuous learning and
dynamic knowledge integration, enabling effective naviga-
tion of complex and evolving environments. Developing AI
systems with analogous capa

Chunk #17
Section: ✅ Related Work
Length: 459 chars
Text preview:
Iterative RAG for Complex Reasoning.
RAG has be-
come a pivotal approach for knowledge-i

## Step 6: Compare Chunking Approaches

| Approach | Section Info | Use Case |
|---|---|---|
| Fixed Size | ❌ No section info | Traditional RAG |
| Section-Aware | ✅ Has section label | Hierarchical RAG |

**Key insight:** Section-aware chunks know WHERE they are in the document.
This is crucial for Hierarchical RAG to work well.

In [9]:
# Same content, different context
print("SAME CHUNK — TWO APPROACHES")
print()

# Find a methodology chunk in fixed chunks
method_fixed = None
for chunk in fixed_chunks:
    if "CHG" in chunk['text'] or "hierarchical graph" in chunk['text'].lower():
        method_fixed = chunk
        break

# Find same content in section chunks
method_section = None
for chunk in section_chunks:
    if "CHG" in chunk['text'] or "hierarchical graph" in chunk['text'].lower():
        method_section = chunk
        break

if method_fixed:
    print("❌ FIXED SIZE CHUNK (Traditional RAG):")
    print(f"   Section: {method_fixed.get('section', 'UNKNOWN')}")
    print(f"   Text: {method_fixed['text'][:200]}...")
    print()

if method_section:
    print("✅ SECTION-AWARE CHUNK (Hierarchical RAG):")
    print(f"   Section: {method_section.get('section', 'UNKNOWN')}")
    print(f"   Text: {method_section['text'][:200]}...")

SAME CHUNK — TWO APPROACHES

❌ FIXED SIZE CHUNK (Traditional RAG):
   Section: UNKNOWN
   Text: Iterative Multi-Granular RAG with Contextual Hierarchical Graph
Yanli Hu1*, Teng Liu1*, Zhuangyi Zhou1, Weixin Zeng1†, Zhen Tan1, Xiang Zhao2
1National Key Laboratory of Information Systems Engineerin...

✅ SECTION-AWARE CHUNK (Hierarchical RAG):
   Section: Abstract
   Text: relationships but incur significant construction costs. To fill
in this gap, we propose MGranRAG, an innovative frame-
work designed to integrate precise local retrieval with struc-
tured global reaso...


In [10]:
import json
import os

os.makedirs("../outputs", exist_ok=True)

# Save fixed chunks
with open("../outputs/fixed_chunks.json", "w") as f:
    json.dump(fixed_chunks, f, indent=2)

# Save section chunks
with open("../outputs/section_chunks.json", "w") as f:
    json.dump(section_chunks, f, indent=2)

print(f"✅ Saved {len(fixed_chunks)} fixed chunks")
print(f"✅ Saved {len(section_chunks)} section-aware chunks")
print(f"✅ Files saved to outputs/ folder")

✅ Saved 100 fixed chunks
✅ Saved 74 section-aware chunks
✅ Files saved to outputs/ folder


## Summary

What we learned:
1. PDF text extraction gives us raw text per page
2. Section detection identifies paper structure
3. Fixed-size chunking = simple but loses context
4. Section-aware chunking = preserves document structure

**Next:** Notebook 02 — Traditional RAG (baseline)